# exp014-017 R2 ensemble inference

**4 Perch distill SED models (5s window、Tucker mel)**:

| Exp | Backbone | LB single |
|---|---|---|
| exp014 | HGNetV2-B0 | 0.907 |
| exp015 | convnext_pico | 0.910 |
| exp016 | regnety_008 | 0.903 |
| exp017 | eca_nfnet_l0 | **0.921** ★ |

**Ensemble**: equal weight average + Gaussian smoothing
**Expected LB**: 0.92-0.94 (single LB の average + diversity bonus)

**Sub source**:
- exp014: kernel `maekeso/birdclef2026-exp014-train-r2`
- exp015: dataset `maekeso/birdclef2026-exp015-weights`
- exp016: dataset `maekeso/birdclef2026-exp016-weights`
- exp017: dataset `maekeso/birdclef2026-exp017-weights`


In [ ]:
!pip install -q timm
import sys, os, time, json, gc
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio.transforms as T
import librosa
import timm
import tqdm.auto as tqdm
from scipy.ndimage import gaussian_filter1d

DEVICE = torch.device("cpu")
torch.set_num_threads(4)
print(f"Python: {sys.version[:50]}, torch: {torch.__version__}, timm: {timm.__version__}")
START = time.time()


In [ ]:
# CFG (Tucker spec、5s) — 3 model ensemble (exp014 excluded: peak ep4 missing)
SR = 32_000
WINDOW_SEC = 5
N_WINDOWS_OUT = 12
N_CLASSES = 234
WINDOW_SAMPLES = SR * WINDOW_SEC

N_MELS = 256
N_FFT = 2048
HOP_LENGTH = 512
F_MIN = 20
F_MAX = 16000

GAUSSIAN_SIGMA = 0.65

def find_dir(candidates):
    for p in candidates:
        if Path(p).exists():
            return Path(p)
    return None

DATA_PATH = find_dir([
    "/kaggle/input/competitions/birdclef-2026",
    "/kaggle/input/birdclef-2026",
])
assert DATA_PATH is not None
TEST_SC_DIR = Path(DATA_PATH) / "test_soundscapes"
SAMPLE_SUB = Path(DATA_PATH) / "sample_submission.csv"

def find_ckpt(roots, candidates):
    for root in roots:
        if root is None or not root.exists(): continue
        for cand in candidates:
            for fp in root.rglob(cand):
                return fp
    return None

# exp015: dataset (convnext_pico R2)
EXP015_ROOT = find_dir([
    "/kaggle/input/birdclef2026-exp015-weights",
    "/kaggle/input/datasets/maekeso/birdclef2026-exp015-weights",
])
EXP015_PTH = find_ckpt([EXP015_ROOT], [
    "r2_ckpt_best_ns22.pth", "r2_ckpt_best_macro.pth",
])

# exp016: dataset (regnety_008 R2)
EXP016_ROOT = find_dir([
    "/kaggle/input/birdclef2026-exp016-weights",
    "/kaggle/input/datasets/maekeso/birdclef2026-exp016-weights",
])
EXP016_PTH = find_ckpt([EXP016_ROOT], [
    "r2_ckpt_best_ns22.pth", "r2_ckpt_best_macro.pth",
])

# exp017: dataset (eca_nfnet_l0 R2)
EXP017_ROOT = find_dir([
    "/kaggle/input/birdclef2026-exp017-weights",
    "/kaggle/input/datasets/maekeso/birdclef2026-exp017-weights",
])
EXP017_PTH = find_ckpt([EXP017_ROOT], [
    "r2_ckpt_best_ns22.pth", "r2_ckpt_best_macro.pth",
])

print(f"exp015: {EXP015_PTH}")
print(f"exp016: {EXP016_PTH}")
print(f"exp017: {EXP017_PTH}")
assert EXP015_PTH and EXP016_PTH and EXP017_PTH, "Missing ckpt"

OUT_DIR_WORK = Path("/kaggle/working")


In [ ]:
# Model architecture (must match training: BirdSEDModel)
class GeMFreqPool(nn.Module):
    def __init__(self, p_init=3.0, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.tensor(float(p_init)))
        self.eps = eps
    def forward(self, x):
        p = self.p.clamp(min=1.0)
        x = x.clamp(min=self.eps).pow(p)
        x = x.mean(dim=2)
        return x.pow(1.0 / p)


class DistillHead(nn.Module):
    def __init__(self, backbone_dim, embed_dim=1536):
        super().__init__()
        self.proj = nn.Linear(backbone_dim, embed_dim)
    def forward(self, feature_map):
        return self.proj(feature_map.mean(dim=[2, 3]))


class BirdSEDModel(nn.Module):
    def __init__(self, backbone_name, num_classes=N_CLASSES, drop_path_rate=0.0, hidden_dim=512,
                 use_perch_distill=True):
        super().__init__()
        self.backbone = timm.create_model(
            backbone_name, pretrained=False, in_chans=1,
            num_classes=0, global_pool="", drop_path_rate=drop_path_rate,
        )
        with torch.no_grad():
            n_tf = WINDOW_SAMPLES // HOP_LENGTH + 1
            dummy = torch.randn(1, 1, N_MELS, n_tf)
            feat = self.backbone(dummy)
            self.backbone_dim = feat.shape[1]

        self.gem_freq = GeMFreqPool(p_init=3.0)
        self.dense = nn.Sequential(
            nn.Dropout(0.25),
            nn.Linear(self.backbone_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
        )
        self.att = nn.Conv1d(hidden_dim, num_classes, kernel_size=1, bias=True)
        self.cla = nn.Conv1d(hidden_dim, num_classes, kernel_size=1, bias=True)
        if use_perch_distill:
            self.distill_head = DistillHead(self.backbone_dim, 1536)

    def forward(self, x, return_framewise=False, return_distill=False):
        h = self.backbone(x)
        h_cls = self.gem_freq(h)
        h_cls = h_cls.permute(0, 2, 1)
        h_cls = self.dense(h_cls)
        h_cls = h_cls.permute(0, 2, 1)
        norm_att = torch.softmax(torch.tanh(self.att(h_cls)), dim=-1)
        framewise = self.cla(h_cls)
        clip_logits = torch.sum(norm_att * framewise, dim=2)
        if return_framewise:
            return clip_logits, framewise.permute(0, 2, 1)  # (B, time, num_class)
        return clip_logits


def sigmoid_np(x):
    return (1.0 / (1.0 + np.exp(-np.clip(x, -50, 50)))).astype(np.float32)


def load_birdsed_model(ckpt_path, backbone_name):
    model = BirdSEDModel(backbone_name=backbone_name).to(DEVICE)
    ckpt = torch.load(str(ckpt_path), weights_only=False, map_location="cpu")
    state = ckpt.get("model_state", ckpt.get("state_dict", ckpt))
    msg = model.load_state_dict(state, strict=False)
    model.eval()
    print(f"  {backbone_name}: missing={len(msg.missing_keys)} unexpected={len(msg.unexpected_keys)}")
    if len(msg.missing_keys) > 5:
        print(f"    sample missing: {list(msg.missing_keys)[:3]}")
    if len(msg.unexpected_keys) > 5:
        print(f"    sample unexpected: {list(msg.unexpected_keys)[:3]}")
    return model


In [ ]:
# Load 3 models (exp014 excluded — peak ep4 ckpt missing from kernel output)
print("Loading 3 models...")
m_exp015 = load_birdsed_model(EXP015_PTH, "convnext_pico.d1_in1k")
m_exp016 = load_birdsed_model(EXP016_PTH, "regnety_008")
m_exp017 = load_birdsed_model(EXP017_PTH, "eca_nfnet_l0")

MODELS = [m_exp015, m_exp016, m_exp017]
MODEL_NAMES = ["exp015_convnext_pico", "exp016_regnety_008", "exp017_eca_nfnet_l0"]

# Weighted: prioritize eca_nfnet_l0 (best single LB 0.921)
WEIGHTS = [0.30, 0.25, 0.45]
print(f"\nEnsemble: {len(MODELS)} models, weights={WEIGHTS}")

# Mel transform (Tucker spec)
mel_transform = T.MelSpectrogram(
    sample_rate=SR, normalized=True, n_fft=N_FFT,
    hop_length=HOP_LENGTH, win_length=N_FFT,
    f_max=F_MAX, n_mels=N_MELS, f_min=F_MIN,
).to(DEVICE)
amp_to_db = T.AmplitudeToDB(top_db=80).to(DEVICE)

def compute_mel(wav_tensor):
    mel = mel_transform(wav_tensor)
    mel = amp_to_db(mel)
    B = mel.size(0)
    for i in range(B):
        mel[i] = (mel[i] - mel[i].mean()) / (mel[i].std() + 1e-6)
    return mel

sample_sub = pd.read_csv(SAMPLE_SUB)
PRIMARY_LABELS = sample_sub.columns[1:].tolist()
assert len(PRIMARY_LABELS) == N_CLASSES


In [ ]:
# Inference function: ensemble 4 models
def infer_file_ensemble(audio_60s):
    """60s audio → 12 chunks × 234 sp probability via 4-model ensemble."""
    # Reshape 60s into 12 × 5s chunks
    chunks = audio_60s[:SR * 60].reshape(N_WINDOWS_OUT, WINDOW_SAMPLES).astype(np.float32)
    # Normalize per chunk
    for ci in range(N_WINDOWS_OUT):
        m = np.abs(chunks[ci]).max()
        if m > 0: chunks[ci] = chunks[ci] / m

    wav_t = torch.from_numpy(chunks).unsqueeze(1).to(DEVICE)  # (12, 1, samples)
    mel = compute_mel(wav_t)  # (12, 1, n_mels, n_frames)

    # Forward through 4 models
    probs_sum = np.zeros((N_WINDOWS_OUT, N_CLASSES), dtype=np.float32)
    with torch.no_grad():
        for w, m in zip(WEIGHTS, MODELS):
            clip_logit, framewise = m(mel, return_framewise=True)
            frame_max = framewise.max(dim=1).values
            p_clip = torch.sigmoid(clip_logit).float().cpu().numpy()
            p_fmax = torch.sigmoid(frame_max).float().cpu().numpy()
            probs = 0.5 * p_clip + 0.5 * p_fmax
            probs_sum += w * probs

    # Smoothing across 12 windows
    probs_smoothed = gaussian_filter1d(probs_sum, sigma=GAUSSIAN_SIGMA, axis=0,
                                       mode="nearest").astype(np.float32)
    return probs_smoothed


In [ ]:
# Run inference on test_soundscapes
test_files = sorted(TEST_SC_DIR.glob("*.ogg"))
print(f"Test files: {len(test_files)}")

rows = []
t0 = time.time()
for fi, fp in enumerate(tqdm.tqdm(test_files, desc="Infer ensemble")):
    try:
        y, _ = librosa.load(str(fp), sr=SR, mono=True)
    except Exception as e:
        print(f"  load fail {fp.name}: {e}")
        y = np.zeros(SR * 60, dtype=np.float32)

    target = SR * 60
    if len(y) < target:
        y = np.pad(y, (0, target - len(y)))
    else:
        y = y[:target]

    probs = infer_file_ensemble(y)

    file_stem = fp.stem
    for k in range(N_WINDOWS_OUT):
        end_sec = (k + 1) * 5
        row_id = f"{file_stem}_{end_sec}"
        rows.append([row_id] + probs[k].tolist())

    if (fi + 1) % 50 == 0 or fi == len(test_files) - 1:
        elapsed = time.time() - t0
        rate = (fi + 1) / max(elapsed, 0.001)
        eta = (len(test_files) - fi - 1) / max(rate, 0.001) / 60
        print(f"  [{fi+1}/{len(test_files)}] {elapsed:.0f}s rate={rate:.2f}f/s eta={eta:.1f}min")

print(f"\nInference DONE: {(time.time()-t0)/60:.1f} min, {len(rows)} rows")

# Build submission
sub_df = pd.DataFrame(rows, columns=["row_id"] + PRIMARY_LABELS)
assert sub_df["row_id"].nunique() == len(sub_df), "Duplicate row_id"
print(f"sub_df: {sub_df.shape}, mean={sub_df[PRIMARY_LABELS].mean().mean():.5f}, max={sub_df[PRIMARY_LABELS].max().max():.4f}")

if len(test_files) > 0 and len(sample_sub) > 0:
    sub_df = sub_df.set_index("row_id").reindex(sample_sub["row_id"]).reset_index()
    sub_df[PRIMARY_LABELS] = sub_df[PRIMARY_LABELS].fillna(0.0)

sub_df.to_csv(OUT_DIR_WORK / "submission.csv", index=False)
print(f"\nSaved: {OUT_DIR_WORK / 'submission.csv'} ({(OUT_DIR_WORK / 'submission.csv').stat().st_size/1e6:.1f} MB)")
print(f"Total time: {(time.time()-START)/60:.1f} min")
